# AutoML Tutorial: H2O AutoML
> **Run this notebook in Google Colab or Jupyter to learn about automated machine learning (AutoML) using H2O AutoML.**

## 1. Introduction
Automated Machine Learning (AutoML) simplifies the end-to-end ML workflow by:
- **Feature preprocessing**
- **Model selection**
- **Hyperparameter optimization**
- **Ensembling**

In this tutorial, we'll focus solely on **H2O AutoML**, a scalable AutoML framework supporting both regression and classification.


At the end, you'll complete an exercise applying AutoML.



## 2. Setup & Installation

In [1]:
!pip install --quiet jedi
!pip install --quiet h2o
!pip install --quiet 'thinc<8.3.6'

The system cannot find the file specified.


In [2]:
!pip install jdk4py


## Regression on California Housing


In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 3. Regression Example: California Housing

In [4]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

In [5]:
import os
import jdk4py
import h2o

# 1. Point the system 'JAVA_HOME' and 'PATH' to the pip-installed Java binary
java_bin_dir = os.path.dirname(jdk4py.JAVA)
os.environ["JAVA_HOME"] = os.path.dirname(java_bin_dir)
os.environ["PATH"] = java_bin_dir + os.pathsep + os.environ["PATH"]

# 2. Initialize H2O safely
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321..... not found.
Attempting to start a local H2O server...
; OpenJDK 64-Bit Server VM Temurin-25.0.2+10 (build 25.0.2+10-LTS, mixed mode)
  Starting server from C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\HP\AppData\Local\Temp\tmp9qv1qh30
  JVM stdout: C:\Users\HP\AppData\Local\Temp\tmp9qv1qh30\h2o_HP_started_from_python.out
  JVM stderr: C:\Users\HP\AppData\Local\Temp\tmp9qv1qh30\h2o_HP_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.


H2O_cluster_uptime:,03 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 5 days
H2O_cluster_name:,H2O_from_python_HP_g89420
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.983 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [6]:
# Load data
data = fetch_california_housing(as_frame=True)
X = data.data
y = data.target.rename('target')

In [7]:
# Create H2OFrame
df = h2o.H2OFrame(pd.concat([X, y], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [8]:
# Split into train/test
train, test = df.split_frame(ratios=[0.8], seed=42)

### 3.1 Run H2O AutoML for Regression

In [9]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    project_name="california_regression"
)
aml_reg.train(x=X.columns.tolist(), y='target', training_frame=train)

AutoML progress: |
03:04:40.158: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_4_AutoML_1_20260728_30440


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    100                100                         282989                 10           10           10            55            449           220.93

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 0.07068300661149503
RMSE: 0.2658627589781898
MAE: 0.18120917114794807
RMSLE: 0.08210427216080109
Mean Residual Deviance: 0.07068300661149503

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 0.20348471185663097
RMSE: 0.4510927973894407
MAE: 0.29284213433945333
RMSLE: 0.13524205346487886
Mean Residual Deviance: 0.20348471185663097

Cross-Validation Metrics Summary: 
                        mean      sd          cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  --------  ----------  ------------  ------------  ------------  ------------  ------------
aic                     nan       0           nan           nan           nan           nan           nan
loglikelihood           nan       0           nan           nan           nan           nan           nan
mae                     0.292841  0.00338635  0.296661      0.287396      0.293333      0.294019      0.292795
mean_residual_deviance  0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
mse                     0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
r2                      0.8475    0.0055502   0.840433      0.849944      0.845547      0.846212      0.855363
residual_deviance       0.203505  0.00921118  0.217368      0.199415      0.205641      0.202794      0.192307
rmse                    0.451024  0.0101703   0.466228      0.44656       0.453476      0.450326      0.438528
rmsle                   0.135246  0.00145376  0.136438      0.134214      0.136616      0.135689      0.133273

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse        training_mae         training_deviance
---  -------------------  ----------  -----------------  -------------------  -------------------  -------------------
     2026-07-28 03:05:57  6.586 sec   0.0                1.1550860447424613   0.9126682923030284   1.3342237707587836
     2026-07-28 03:05:57  6.730 sec   5.0                0.805071829765977    0.6296027937088264   0.6481406510827383
     2026-07-28 03:05:57  6.860 sec   10.0               0.602250749033284    0.45921175920543444  0.3627059647111517
     2026-07-28 03:05:57  6.983 sec   15.0               0.48995730387890285  0.36044403364806166  0.24005815962428353
     2026-07-28 03:05:57  7.094 sec   20.0               0.4331452215860766   0.30982096116895946  0.18761478298285142
     2026-07-28 03:05:57  7.189 sec   25.0               0.3968391157329822   0.2769064720926289   0.15748128377573525
     2026-07-28 03:05:57  7.284 sec   30.0               0.3741522705393597   0.25627071575137306  0.1399899215497582
     2026-07-28 03:05:58  7.379 sec   35.0               0.35844112136034767  0.24324635480521686  0.1284800374820635
     2026-07-28 03:05:58  7.473 sec   40.0               0.34463815160880085  0.2327461465752868   0.11877545554433079
     2026-07-28 03:05:58  7.568 sec   45.0               0.3338572144190766   0.2247170105589096   0.11146063961966529
---  ---                  ---         ---                ---                  ---                  ---
     2026-07-28 03:05:58  7.742 sec   55.0               0.31870769879870675  0.21496977659685737  0.10157459727356719
     2026-07-28 03:05:58  7.821 sec   60.0               0.

In [10]:
# Show leaderboard
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


                                     model_id      rmse       mse       mae  \
0               GBM_4_AutoML_1_20260728_30440  0.451093  0.203485  0.292842   
1               GBM_3_AutoML_1_20260728_30440  0.451954  0.204262  0.297087   
2               GBM_2_AutoML_1_20260728_30440  0.456719  0.208592  0.301624   
3               GBM_1_AutoML_1_20260728_30440  0.458936  0.210623  0.303082   
4  GBM_grid_1_AutoML_1_20260728_30440_model_5  0.461007  0.212527  0.300461   

      rmsle  mean_residual_deviance  
0  0.135242                0.203485  
1  0.136120                0.204262  
2  0.137787                0.208592  
3  0.138777                0.210623  
4  0.138387                0.212527  


In [11]:
# Evaluate on test
perf_reg = aml_reg.leader.model_performance(test)
print(f"H2O Regression R²: {perf_reg.r2():.4f}")
print(f"H2O Regression RMSE: {perf_reg.rmse():.4f}")

H2O Regression R²: 0.8569
H2O Regression RMSE: 0.4346


## 4. Classification Example: Breast Cancer Dataset

In [12]:
from sklearn.datasets import load_breast_cancer

In [13]:
# Load and prepare dataset
data_cls = load_breast_cancer(as_frame=True)
Xc = data_cls.data
yc = data_cls.target.rename('target')

In [14]:
df_cls = h2o.H2OFrame(pd.concat([Xc, yc], axis=1))
train_cls, test_cls = df_cls.split_frame(ratios=[0.7], seed=42)

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


### 4.1 Run H2O AutoML for Classification

In [15]:
aml_cls = H2OAutoML(
    max_runtime_secs=300,
    max_models=20,
    seed=42,
    nfolds=5,
    balance_classes=True,
    project_name="breast_cancer_classification"
)
aml_cls.train(x=Xc.columns.tolist(), y='target', training_frame=train_cls)

AutoML progress: |
03:10:00.599: AutoML: XGBoost is not available; skipping it.
03:10:00.608: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.


03:10:01.13: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
03:10:01.856: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical before training.

█
03:10:03.292: _response param, We have detected that your response column has only 2 unique values (0/1). If you wish to train a binary model instead of a regression model, convert your target column to categorical be

key,value
Stacking strategy,cross_validation
Number of base models (used / total),10/28
# GBM base models (used / total),8/25
# DRF base models (used / total),1/2
# GLM base models (used / total),1/1
Metalearner algorithm,GLM
Metalearner fold assignment scheme,Random
Metalearner nfolds,5
Metalearner fold_column,None
Custom metalearner hyperparameters,None


In [16]:
# Show leaderboard
df_leader_cls = aml_cls.leaderboard.as_data_frame()
print(df_leader_cls.head())

                                            model_id      rmse       mse  \
0  StackedEnsemble_AllModels_1_AutoML_2_20260728_...  0.181630  0.032989   
1  StackedEnsemble_BestOfFamily_1_AutoML_2_202607...  0.182539  0.033320   
2        GBM_grid_1_AutoML_2_20260728_31000_model_25  0.185965  0.034583   
3        GBM_grid_1_AutoML_2_20260728_31000_model_14  0.186716  0.034863   
4        GBM_grid_1_AutoML_2_20260728_31000_model_23  0.188528  0.035543   

        mae     rmsle  mean_residual_deviance  
0  0.097113  0.131007                0.032989  
1  0.104637  0.131654                0.033320  
2  0.096905  0.131893                0.034583  
3  0.089201  0.134120                0.034863  
4  0.090711  0.133354                0.035543  


C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


In [17]:
# Evaluate on test
perf_cls = aml_cls.leader.model_performance(test_cls)
print(f"H2O Classification: {perf_cls}")

H2O Classification: ModelMetricsRegressionGLM: stackedensemble
** Reported on test data. **

MSE: 0.03556103996032552
RMSE: 0.18857635047991972
MAE: 0.09881519448380302
RMSLE: 0.12886513962124194
Mean Residual Deviance: 0.03556103996032552
R^2: 0.8421303082383419
Null degrees of freedom: 177
Residual degrees of freedom: 167
Null deviance: 40.43221198186829
Residual deviance: 6.329865112937943
AIC: -64.75570516846703


MSE (Mean Squared Error):
- Measures the average squared difference between predicted and actual values. Lower values indicate better performance.


RMSE (Root Mean Squared Error):
- The square root of MSE, providing an error metric in the same units as the target variable. Lower values are better.


MAE (Mean Absolute Error):
-Measures the average absolute difference between predicted and actual values. Lower values indicate better performance.


RMSLE (Root Mean Squared Logarithmic Error):
- Similar to RMSE but uses the logarithm of the values, making it more robust to outliers. Lower values are better.


Mean Residual Deviance:
- Measures the goodness of fit of the model. Lower values indicate a better fit.


R² (Coefficient of Determination):
- Represents the proportion of variance explained by the model. Values closer to 1 indicate better performance.


Null Degrees of Freedom:
- The number of observations minus 1.


Residual Degrees of Freedom:
- The number of observations minus the number of parameters estimated.


Null Deviance:
- The deviance of the null model (model with no predictors).


Residual Deviance:
- The deviance of the fitted model. Lower values indicate a better fit.


AIC (Akaike Information Criterion):
- A measure of model quality, balancing goodness of fit and model complexity. Lower values indicate a better model.

## 5. Interpreting Results
- **Leaderboard** displays model ranking by default metric.
- Use `model_performance` to compute custom metrics (RMSE, R², AUC, accuracy, etc.).
- Top models are automatically ensembled by H2O's Stacked Ensemble.

## 6. AutoML Best Practices
- **Set time and model limits** (`max_runtime_secs`, `max_models`) to control cost and runtime.
- **Use cross-validation** (`nfolds`) for robust performance estimates.
- **Balance classes** for imbalanced classification tasks.
- **Review variable importance** on the leader model: `aml.leader.varimp()`.
- **Save and deploy** the best model: `h2o.save_model(aml.leader, path='best_model')`.

## 7. Exercise: Custom Dataset AutoML

**Task:** Apply H2O AutoML to a custom dataset.

1. Load any tabular dataset (CSV or from `sklearn.datasets`).
2. Decide whether it's a regression or classification task.
3. Initialize H2O and convert to `H2OFrame`.
4. Split into appropriate train/test ratios.
5. Run `H2OAutoML` with:
   - `max_runtime_secs=300`
   - `max_models=15`
   - `nfolds=5`
   - `balance_classes=True` (if classification)
6. Display the leaderboard and evaluate on the test set using relevant metrics.
7. Save the leaderboard to a pandas DataFrame and export it as `leaderboard.csv`

In [47]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.datasets

In [48]:
data = fetch_openml(name='elevators', version=1)

In [49]:
data.DESCR

'This data set is also obtained from the task of controlling a F16\naircraft, although the target variable and attributes are different\nfrom the ailerons domain. In this case the goal variable is related to\nan action taken on the elevators of the aircraft.\n\nDownloaded from openml.org.'

In [50]:
x=data.data
y=data.target

In [51]:
import os
import jdk4py
import h2o

# 1. Point the system 'JAVA_HOME' and 'PATH' to the pip-installed Java binary
java_bin_dir = os.path.dirname(jdk4py.JAVA)
os.environ["JAVA_HOME"] = os.path.dirname(java_bin_dir)
os.environ["PATH"] = java_bin_dir + os.pathsep + os.environ["PATH"]

# 2. Initialize H2O safely
h2o.init(max_mem_size="2G", nthreads=-1)

Checking whether there is an H2O instance running at http://localhost:54321. connected.


H2O_cluster_uptime:,31 mins 32 secs
H2O_cluster_timezone:,Asia/Karachi
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.11
H2O_cluster_version_age:,2 months and 5 days
H2O_cluster_name:,H2O_from_python_HP_g89420
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,1.784 Gb
H2O_cluster_total_cores:,8
H2O_cluster_allowed_cores:,8
H2O_cluster_status:,"locked, healthy"


In [52]:
df = h2o.H2OFrame(pd.concat([data.data, data.target], axis=1))

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


In [56]:
df.head(1)

climbRate,Sgz,p,q,curRoll,absRoll,diffClb,diffRollRate,diffDiffClb,SaTime1,SaTime2,SaTime3,SaTime4,diffSaTime1,diffSaTime2,diffSaTime3,diffSaTime4,Sa,Goal
118,-55,-0.28,-0.08,-0.2,-11,11,0.005,-0.2,-0.001,-0.001,-0.001,-0.001,0,0,0,0,-0.001,0.031


In [53]:
train, test = df.split_frame(ratios=[0.8], seed=42)


In [58]:
aml_reg = H2OAutoML(
    max_runtime_secs=300,
    max_models=15,
    seed=42,
    nfolds=5,
    project_name="Elevator"
)
aml_reg.train(x=data.data.columns.tolist(), y='Goal', training_frame=train)

AutoML progress: |█
03:38:59.52: AutoML: XGBoost is not available; skipping it.
03:43:06.764: New models will be added to existing leaderboard Elevator@@Goal (leaderboard frame=null) with already 7 models.
03:43:06.764: AutoML: XGBoost is not available; skipping it.

██████████████████████████████████████████████████████████████| (done) 100%


Model Details
=============
H2OGradientBoostingEstimator : Gradient Boosting Machine
Model Key: GBM_2_AutoML_6_20260728_33859


Model Summary: 
    number_of_trees    number_of_internal_trees    model_size_in_bytes    min_depth    max_depth    mean_depth    min_leaves    max_leaves    mean_leaves
--  -----------------  --------------------------  ---------------------  -----------  -----------  ------------  ------------  ------------  -------------
    180                180                         171863                 7            7            7             31            111           71.3555

ModelMetricsRegression: gbm
** Reported on train data. **

MSE: 2.096264485627946e-06
RMSE: 0.0014478482260333596
MAE: 0.0010737207345105568
RMSLE: 0.0014144601323235043
Mean Residual Deviance: 2.096264485627946e-06

ModelMetricsRegression: gbm
** Reported on cross-validation data. **

MSE: 4.874329942157831e-06
RMSE: 0.0022077884731463364
MAE: 0.0015959844915571942
RMSLE: 0.002151209491723927
Mean Residual Deviance: 4.874329942157831e-06

Cross-Validation Metrics Summary: 
                        mean         sd           cv_1_valid    cv_2_valid    cv_3_valid    cv_4_valid    cv_5_valid
----------------------  -----------  -----------  ------------  ------------  ------------  ------------  ------------
aic                     nan          0            nan           nan           nan           nan           nan
loglikelihood           nan          0            nan           nan           nan           nan           nan
mae                     0.00159598   2.11224e-05  0.00158235    0.00162188    0.00161297    0.00159166    0.00157107
mean_residual_deviance  4.87433e-06  3.9568e-08   4.93343e-06   4.87642e-06   4.86226e-06   4.82318e-06   4.87636e-06
mse                     4.87433e-06  3.9568e-08   4.93343e-06   4.87642e-06   4.86226e-06   4.82318e-06   4.87636e-06
r2                      0.891028     0.00227504   0.892293      0.893069      0.888009      0.889184      0.892583
residual_deviance       4.87433e-06  3.9568e-08   4.93343e-06   4.87642e-06   4.86226e-06   4.82318e-06   4.87636e-06
rmse                    0.00220777   8.95596e-06  0.00222113    0.00220826    0.00220505    0.00219617    0.00220825
rmsle                   0.0021512    7.74331e-06  0.00216225    0.00215261    0.00215102    0.00214056    0.00214956

Scoring History: 
     timestamp            duration    number_of_trees    training_rmse          training_mae           training_deviance
---  -------------------  ----------  -----------------  ---------------------  ---------------------  ----------------------
     2026-07-28 03:40:16  12.325 sec  0.0                0.006689550317837822   0.004569838799294875   4.4750083454884107e-05
     2026-07-28 03:40:16  12.467 sec  5.0                0.004826559616393075   0.0033730502776237064  2.3295677730596466e-05
     2026-07-28 03:40:16  12.563 sec  10.0               0.003826332799699864   0.002704569674743268   1.4640822694059001e-05
     2026-07-28 03:40:16  12.673 sec  15.0               0.003246930256051237   0.002307816916784664   1.054255608766095e-05
     2026-07-28 03:40:16  12.783 sec  20.0               0.00289522419389182    0.002053939135687877   8.38232313289654e-06
     2026-07-28 03:40:17  12.880 sec  25.0               0.002686033321600722   0.0019005401509926449  7.214775004749409e-06
     2026-07-28 03:40:17  13.000 sec  30.0               0.0025072537185743507  0.0017735191212865033  6.2863212093049094e-06
     2026-07-28 03:40:17  13.111 sec  35.0               0.0023697326962526088  0.0016796903061524692  5.615633051688659e-06
     2026-07-28 03:40:17  13.232 sec  40.0               0.0022583264096450854  0.001602680605835514   5.100038172500462e-06
     2026-07-28 03:40:17  13.347 sec  45.0               0.0021671966251153633  0.0015438287346347865  4.696741211911421e-06
---  ---                  ---         ---                ---                    ---                    ---
     2026-07-2

In [59]:
df_leader_reg = aml_reg.leaderboard.as_data_frame()
print(df_leader_reg.head())

C:\Users\HP\miniconda3\envs\ds_learning\Lib\site-packages\h2o\frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"


                                     model_id      rmse       mse       mae  \
0               GBM_2_AutoML_6_20260728_33859  0.002208  0.000005  0.001596   
1               GBM_7_AutoML_7_20260728_34306  0.002208  0.000005  0.001596   
2  GBM_grid_1_AutoML_7_20260728_34306_model_2  0.002228  0.000005  0.001604   
3               GBM_8_AutoML_7_20260728_34306  0.002240  0.000005  0.001609   
4               GBM_3_AutoML_6_20260728_33859  0.002240  0.000005  0.001609   

      rmsle  mean_residual_deviance  
0  0.002151                0.000005  
1  0.002151                0.000005  
2  0.002171                0.000005  
3  0.002181                0.000005  
4  0.002181                0.000005  


In [61]:
perf = aml_reg.leader.model_performance(test)
print(f"H2O Classification: {perf}")

H2O Classification: ModelMetricsRegression: gbm
** Reported on test data. **

MSE: 4.405246357575565e-06
RMSE: 0.002098867875206909
MAE: 0.0015559464476368086
RMSLE: 0.002045972952726422
Mean Residual Deviance: 4.405246357575565e-06


In [63]:
df_leader_reg.shape

(19, 6)

In [65]:
df_leader_reg.to_csv('Elevator_dataset_leaderboad.csv', index=False)